# Rocket Launch Data Preparation



This notebook builds the cleaned launch-level dataset that the rest of the project uses for EDA and modeling. The workflow starts from the raw launch, mission, configuration, family, location, and hourly weather tables; standardizes the launch records; narrows the analysis to U.S. launch sites with usable weather coverage; engineers history, cadence, and weather features; and writes the final prepared tables to `data/derived/`.



A useful way to read the notebook is as a pipeline:

1. Load the raw source tables and confirm their size.

2. Collapse mission/config/location data down to one row per launch.

3. Normalize U.S. launch facilities so each launch can be matched to the correct weather file.

4. Join each launch to the nearest hourly weather observation and derive model-ready features.

5. Audit feature coverage and export the prepared outputs.



### What This Setup Cell Does

This cell imports the libraries used throughout the notebook, sets display and plotting defaults, and defines the shared constants that later cells rely on.

Why these pieces matter:
- `DATA_DIR` and `OUTPUT_DIR` centralize file paths so the rest of the notebook does not hard-code directories repeatedly.
- The weather column lists define which weather variables to keep from the raw LCD files and separate numeric fields from text fields that need different cleaning logic.
- `WEATHER_FILE_MAP` is the bridge between launch sites and weather files. It says which CSV belongs to each normalized facility and what UTC offset to use so launch timestamps can be converted into the same local clock used by the weather station records.

That last point is especially important for the merge later: the launch table stores timestamps in UTC, while the weather files are interpreted in local standard time, so the notebook needs both the site-to-file mapping and the local time offset to align the two sources correctly.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", palette="deep")

DATA_DIR = Path("data")
OUTPUT_DIR = DATA_DIR / "derived"
OUTPUT_DIR.mkdir(exist_ok=True)

WEATHER_NUMERIC_COLUMNS = ['HourlyAltimeterSetting', 'HourlyDryBulbTemperature', 'HourlyDewPointTemperature', 'HourlyRelativeHumidity', 'HourlyPrecipitation', 'HourlyVisibility', 'HourlyStationPressure', 'HourlySeaLevelPressure', 'HourlyWetBulbTemperature', 'HourlyWindSpeed', 'HourlyWindGustSpeed', 'HourlyWindDirection']
WEATHER_TEXT_COLUMNS = ['HourlyPresentWeatherType', 'HourlySkyConditions']
SHORT_DURATION_PRECIP_COLUMNS = ['ShortDurationPrecipitationValue005', 'ShortDurationPrecipitationValue010', 'ShortDurationPrecipitationValue015', 'ShortDurationPrecipitationValue020', 'ShortDurationPrecipitationValue030', 'ShortDurationPrecipitationValue045', 'ShortDurationPrecipitationValue060', 'ShortDurationPrecipitationValue080', 'ShortDurationPrecipitationValue100', 'ShortDurationPrecipitationValue120', 'ShortDurationPrecipitationValue150', 'ShortDurationPrecipitationValue180']
WEATHER_FILE_MAP = {'Cape Canaveral Space Force Station': ('cape_canaveral_sfs/cape_canaveral_sfs.csv', -5), 'Kennedy Space Center': ('kennedy_sc/kennedy_sc.csv', -5), 'Vandenberg Space Force Base': ('vandenberg_sfb/vandenberg_sfb.csv', -8), 'Wallops Flight Facility': ('wallops_flight_facility/wallops_flight_facility.csv', -5), 'Pacific Spaceport Complex Alaska': ('pacific_spaceport_alaska/pacific_spaceport_alaska.csv', -9), 'Pacific Missile Range Facility': ('pacific_missile_range/pacific_missile_range.csv', -10), 'Mojave Air and Space Port': ('mojave_air_space_port/mojave_air_space_port.csv', -8), 'Edwards Air Force Base': ('edwards_afb/edwards_afb.csv', -8), 'China Lake': ('china_lake/china_lake.csv', -8)}


## 1. Load Source Tables



This section pulls the six raw project tables into pandas. The goal is not transformation yet; it is to establish the core inputs that later cells will combine into a single launch-level modeling table.



### What This Loading Cell Does

This cell reads the six raw CSV tables into pandas dataframes and creates a quick `dataset_shapes` summary.

Why this matters:
- The notebook uses multiple source tables because no single raw file contains everything needed for modeling.
- The shape summary is a lightweight sanity check that the files were read successfully and still look roughly like expected inputs before any merges happen.
- Looking at row and column counts early makes it easier to spot broken paths, incomplete files, or schema changes before they propagate into downstream feature engineering.


In [2]:
companies = pd.read_csv(DATA_DIR / "Companies.csv")
configs = pd.read_csv(DATA_DIR / "Configs.csv")
families = pd.read_csv(DATA_DIR / "Families.csv")
launches = pd.read_csv(DATA_DIR / "Launches.csv")
locations = pd.read_csv(DATA_DIR / "Locations.csv")
missions = pd.read_csv(DATA_DIR / "Missions.csv")

dataset_shapes = pd.DataFrame(
    {
        "dataset": ["Companies", "Configs", "Families", "Launches", "Locations", "Missions"],
        "rows": [len(companies), len(configs), len(families), len(launches), len(locations), len(missions)],
        "columns": [companies.shape[1], configs.shape[1], families.shape[1], launches.shape[1], locations.shape[1], missions.shape[1]],
    }
)
dataset_shapes


,dataset,rows,columns
0,Companies,59,3
1,Configs,480,13
2,Families,205,8
3,Launches,6168,16
4,Locations,145,14
5,Missions,7450,4


## 2. Launch-Level Preparation



The raw files are stored at different granularities, so this section reshapes them to a consistent launch-level view. It also parses numeric values that are stored as text and adds basic calendar fields that are useful later for grouping, seasonality analysis, and engineered features.



### What This Launch-Preparation Cell Does

This cell converts the raw launch table into a richer launch-level dataframe called `launch_df`.

How it works:
- `parse_numeric_text` pulls numeric values out of text fields like thrust, payload, and dimensions so they can be analyzed quantitatively.
- The launch timestamp is parsed into `launch_time_utc`, then expanded into date parts such as year, month, month name, and decade.
- The mission table is aggregated up to one row per launch so each launch gets totals like payload count and mission mass instead of remaining at the individual mission-entry level.
- The config table is merged with rocket family information, numeric config fields are cleaned, and family success rate is converted from a percent string into a numeric variable.
- `launch_df` is then merged with location metadata, mission aggregates, and configuration features so the notebook ends this section with one row per launch and a much wider set of descriptive variables.

Why this matters:
- The downstream model needs one observation per launch, not separate mission, configuration, and location tables.
- Calendar fields help capture seasonality and long-run era effects.
- Mission/config/location merges add operational and physical context that could plausibly affect launch outcomes, such as vehicle family, payload capacity, or launch-site metadata.
- The final `prep_summary` acts as another sanity check by reporting missing timestamps, location coverage, and how often the supporting merges succeeded.


In [3]:
def parse_numeric_text(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.extract(r"([-+]?[0-9]*\.?[0-9]+)")[0],
        errors="coerce",
    )


launch_df = launches.copy()
launch_df["launch_time_utc"] = pd.to_datetime(launch_df["Launch Time"], utc=True, errors="coerce")
launch_df["launch_date"] = launch_df["launch_time_utc"].dt.date
launch_df["launch_year"] = launch_df["launch_time_utc"].dt.year
launch_df["launch_month"] = launch_df["launch_time_utc"].dt.month
launch_df["launch_month_name"] = launch_df["launch_time_utc"].dt.month_name()
launch_df["launch_decade"] = (launch_df["launch_year"] // 10) * 10

mission_agg = (
    missions.groupby("Launch Id")
    .agg(payload_count=("Payloads", "sum"), mission_mass=("Mass", "sum"), mission_rows=("No", "count"))
    .reset_index()
)

config_features = configs.merge(
    families[["Family Id", "Family", "Success Rate"]],
    on="Family Id",
    how="left",
).copy()

for col in [
    "Liftoff Thrust", "Payload to LEO", "Payload to GTO", "Stages",
    "Strap-ons", "Rocket Height", "Fairing Diameter", "Fairing Height",
]:
    if col in config_features.columns:
        config_features[col] = parse_numeric_text(config_features[col])

config_features["family_success_rate_pct"] = pd.to_numeric(
    config_features["Success Rate"].astype(str).str.rstrip("%"),
    errors="coerce",
)

launch_df = launch_df.merge(
    locations[
        [
            "Orig_Addr", "Country", "Country_Code", "Operator", "Launch Site",
            "Comb Launch Site", "Lat", "Lon", "Comb Launch Site Lat", "Comb Launch Site Lon",
        ]
    ],
    left_on="Location",
    right_on="Orig_Addr",
    how="left",
)
launch_df = launch_df.merge(mission_agg, on="Launch Id", how="left")
launch_df = launch_df.merge(
    config_features[
        [
            "Config", "Status", "Liftoff Thrust", "Payload to LEO", "Payload to GTO",
            "Stages", "Strap-ons", "Rocket Height", "Fairing Diameter", "Fairing Height",
            "Family", "family_success_rate_pct",
        ]
    ],
    left_on="Rocket Name",
    right_on="Config",
    how="left",
)
launch_df = launch_df.rename(
    columns={
        "Payload to LEO": "config_payload_leo",
        "Payload to GTO": "config_payload_gto",
        "Status": "config_status",
        "Liftoff Thrust": "config_liftoff_thrust",
        "Stages": "config_stages",
        "Strap-ons": "config_strap_ons",
        "Rocket Height": "config_rocket_height",
        "Fairing Diameter": "config_fairing_diameter",
        "Fairing Height": "config_fairing_height",
        "Family": "rocket_family",
    }
)

prep_summary = pd.DataFrame(
    {
        "metric": [
            "total launches", "missing launch timestamps", "missing locations",
            "unique raw locations", "unique combined launch sites", "mission aggregates available",
            "config family available",
        ],
        "value": [
            len(launch_df), int(launch_df["launch_time_utc"].isna().sum()),
            int(launch_df["Location"].isna().sum()), int(launch_df["Location"].nunique()),
            int(launch_df["Comb Launch Site"].nunique()), int(launch_df["payload_count"].notna().sum()),
            int(launch_df["rocket_family"].notna().sum()),
        ],
    }
)
prep_summary


,metric,value
0,total launches,6168
1,missing launch timestamps,0
2,missing locations,0
3,unique raw locations,137
4,unique combined launch sites,39
5,mission aggregates available,5557
6,config family available,6168


## 3. U.S. Scope and Facility Normalization



Weather coverage is only available for a defined set of U.S. launch facilities, so this section limits the analysis to U.S. launches and maps messy location strings onto a smaller set of canonical facility groups. That normalization step is important because the weather files are organized by site, not by every raw address variant found in the launch table.



### What This U.S. Facility Cell Does

This cell narrows the prepared launch table to U.S. launches and standardizes launch-site labels into a smaller set of facility groups.

How it works:
- `infer_us_facility_group` scans each raw location string for recognizable site names and maps many string variants onto one canonical facility label.
- The filter `Country_Code == "US"` creates `us_launches`, because the weather workflow in this project only has curated site files for U.S. facilities.
- The new `facility_group` column becomes the key used to decide which weather CSV each launch should be matched against.

Why this matters:
- Raw location strings are often too inconsistent to join directly to weather data.
- The merge later is site-specific, so launches from Kennedy, Cape Canaveral, Vandenberg, and other facilities must be grouped consistently first.
- Restricting to U.S. launches avoids false joins and keeps the prepared dataset aligned with the weather coverage the team actually has.
- The `us_summary` output checks the resulting scope in terms of volume, date range, and site diversity.


In [4]:
def infer_us_facility_group(location: str) -> str:
    location = "" if pd.isna(location) else str(location)
    if "Kennedy Space Center" in location:
        return "Kennedy Space Center"
    if "Cape Canaveral SFS" in location:
        return "Cape Canaveral Space Force Station"
    if "Vandenberg SFB" in location:
        return "Vandenberg Space Force Base"
    if "Wallops Flight Facility" in location:
        return "Wallops Flight Facility"
    if "Pacific Spaceport Complex" in location:
        return "Pacific Spaceport Complex Alaska"
    if "Pacific Missile Range Facility" in location or "Kauai" in location:
        return "Pacific Missile Range Facility"
    if "Mojave Air and Space Port" in location:
        return "Mojave Air and Space Port"
    if "Edwards AFB" in location:
        return "Edwards Air Force Base"
    if "China Lake" in location:
        return "China Lake"
    return "Other U.S. site"


us_launches = launch_df.loc[launch_df["Country_Code"] == "US"].copy()
us_launches["facility_group"] = us_launches["Location"].apply(infer_us_facility_group)

us_summary = pd.DataFrame(
    {
        "metric": [
            "U.S. launches", "U.S. min launch date", "U.S. max launch date",
            "unique U.S. raw location strings", "unique U.S. combined launch sites",
        ],
        "value": [
            len(us_launches), str(us_launches["launch_date"].min()), str(us_launches["launch_date"].max()),
            int(us_launches["Location"].nunique()), int(us_launches["Comb Launch Site"].nunique()),
        ],
    }
)
us_summary


,metric,value
0,U.S. launches,1784
1,U.S. min launch date,1957-12-06
2,U.S. max launch date,2021-12-21
3,unique U.S. raw location strings,50
4,unique U.S. combined launch sites,8


## 4. Launch-Weather Merge and Engineered Features



This is the core feature-building section. It cleans each site-level weather file, matches launches to the nearest hourly observation in local site time, and then creates outcome, operational-history, weather-condition, interaction, and missingness features that can be used in downstream analysis and modeling.



### What This Weather-Merge and Feature-Engineering Cell Does

This is the notebooks main transformation cell. It cleans each hourly weather file, aligns each launch with the closest weather observation, and creates the engineered variables used later in analysis and modeling.

How the weather merge works:
- `load_best_hourly_weather` reads one sites raw LCD weather file, keeps only the relevant columns, converts weather measurements from text to numeric, parses observation timestamps, and removes duplicate hourly records by keeping the most information-rich entry for each hour.
- The function also derives weather-condition flags from coded text fields, such as rain, fog, thunder, broken/overcast cloud cover, and maximum short-duration precipitation.
- Inside the facility loop, launches are split by `facility_group` so each site is merged only with its own weather file.
- Launch times are converted from UTC to local standard time using the offset stored in `WEATHER_FILE_MAP`. This is necessary because weather observations are timestamped according to the local station clock, so merging in UTC would misalign the launch and weather context.
- `pd.merge_asof(..., direction="nearest", tolerance="2h")` performs a nearest-neighbor time join. For each launch, it finds the closest hourly weather observation within a two-hour window.

Why the merge is designed this way:
- Launches and weather stations are not recorded on the exact same timestamp schedule, so an exact join would miss many valid matches.
- Using the nearest hourly observation preserves more launches while still enforcing a reasonable time window.
- The tolerance protects against attaching stale or unrelated weather to a launch when station coverage is sparse.
- The coverage summary records match rates and typical time differences by site so the team can judge whether the merge quality is acceptable.

What feature engineering happens after the merge:
- Outcome targets are created, including binary success/failure indicators and separate flags for failure subtypes.
- Rocket economics and payload fields are converted to numeric form, and smaller categories are grouped so they are easier to model.
- Calendar-style features are added, such as launch season, local launch hour, time-of-day bins, and era groups.
- Direct weather indicators are created, including precipitation, high wind, low visibility, high humidity, and dewpoint depression.
- Interaction features combine weather signals that may be more informative together than separately, such as wind with low visibility or rain with low visibility.
- Missingness flags explicitly record where important engineering or weather fields are absent, which can itself carry information and helps later models handle incomplete data more cleanly.
- `add_prelaunch_history` builds history-based features for rocket families, organizations, specific rocket configurations, and sites. These include prior launch counts, prior successes, pre-launch success rates, years since first launch, and days since the previous launch.
- Site-relative z-scores standardize weather variables within each facility so the model can distinguish ?unusually windy for this site? from conditions that are normal for a location with a different baseline climate.

Why these engineered features are useful:
- Launch outcome is unlikely to depend on raw weather alone; it may also depend on operational maturity, cadence, and whether current conditions are extreme relative to what a site usually experiences.
- History features capture learning effects and organizational reliability without leaking future information, because each launch only uses data available before that launch.
- Interaction and z-score features help the team test more realistic hypotheses than simple raw-variable thresholds, especially when comparing sites with very different normal weather patterns.


In [5]:
def clean_lcd_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.extract(r"([-+]?[0-9]*\.?[0-9]+)")[0],
        errors="coerce",
    )


def load_best_hourly_weather(rel_path: str) -> pd.DataFrame:
    weather_raw = pd.read_csv(DATA_DIR / rel_path, low_memory=False)
    keep_cols = ["DATE", "REPORT_TYPE"] + [
        c
        for c in WEATHER_NUMERIC_COLUMNS + WEATHER_TEXT_COLUMNS + SHORT_DURATION_PRECIP_COLUMNS
        if c in weather_raw.columns
    ]
    weather = weather_raw[keep_cols].copy()
    for col in [c for c in WEATHER_NUMERIC_COLUMNS + SHORT_DURATION_PRECIP_COLUMNS if c in weather.columns]:
        weather[col] = clean_lcd_numeric(weather[col])
    weather["weather_obs_time_lstd"] = pd.to_datetime(weather["DATE"], errors="coerce")
    weather = weather.dropna(subset=["weather_obs_time_lstd"]).copy()
    numeric_cols = [c for c in WEATHER_NUMERIC_COLUMNS if c in weather.columns]
    text_cols = [c for c in WEATHER_TEXT_COLUMNS if c in weather.columns]
    weather["hourly_nonnulls"] = weather[numeric_cols].notna().sum(axis=1)
    for col in text_cols:
        weather["hourly_nonnulls"] += weather[col].notna().astype(int)
    weather = (
        weather.loc[weather["hourly_nonnulls"] > 0]
        .sort_values(["weather_obs_time_lstd", "hourly_nonnulls"], ascending=[True, False])
        .drop_duplicates(subset=["weather_obs_time_lstd"], keep="first")
        .sort_values("weather_obs_time_lstd")
        .reset_index(drop=True)
    )
    present_weather = weather.get("HourlyPresentWeatherType", pd.Series("", index=weather.index)).fillna("").astype(str)
    sky_conditions = weather.get("HourlySkyConditions", pd.Series("", index=weather.index)).fillna("").astype(str)
    weather["present_weather_rain_flag"] = present_weather.str.contains(r"RA|DZ|SH", regex=True)
    weather["present_weather_fog_flag"] = present_weather.str.contains(r"FG|BR|HZ", regex=True)
    weather["present_weather_thunder_flag"] = present_weather.str.contains(r"TS", regex=True)
    weather["cloud_cover_broken_or_overcast_flag"] = sky_conditions.str.contains(r"BKN|OVC", regex=True)
    available_short_duration_cols = [c for c in SHORT_DURATION_PRECIP_COLUMNS if c in weather.columns]
    weather["short_duration_precip_max"] = (
        weather[available_short_duration_cols].max(axis=1, skipna=True)
        if available_short_duration_cols else float("nan")
    )
    return weather


def add_prelaunch_history(df: pd.DataFrame, group_col: str, prefix: str) -> pd.DataFrame:
    valid = df[group_col].notna()
    grp = df.loc[valid].groupby(group_col, sort=False)
    df.loc[valid, f"{prefix}_prior_launches"] = grp.cumcount()
    df.loc[valid, f"{prefix}_prior_successes"] = grp["launch_success_binary"].cumsum() - df.loc[valid, "launch_success_binary"]
    df[f"{prefix}_success_rate_prelaunch"] = df[f"{prefix}_prior_successes"] / df[f"{prefix}_prior_launches"].replace(0, np.nan)
    first_time = grp["launch_time_utc"].transform("min")
    prev_time = grp["launch_time_utc"].shift(1)
    df.loc[valid, f"{prefix}_years_since_first_launch"] = (
        (df.loc[valid, "launch_time_utc"] - first_time).dt.total_seconds() / (365.25 * 24 * 3600)
    )
    df.loc[valid, f"days_since_previous_launch_{prefix}"] = (
        (df.loc[valid, "launch_time_utc"] - prev_time).dt.total_seconds() / (24 * 3600)
    )
    return df


weather_merges = []
weather_coverage_rows = []
for facility, (rel_path, utc_offset_hours) in WEATHER_FILE_MAP.items():
    facility_launches = us_launches.loc[us_launches["facility_group"] == facility].copy()
    facility_launches["launch_time_lstd"] = (
        facility_launches["launch_time_utc"] + pd.to_timedelta(utc_offset_hours, unit="h")
    ).dt.tz_localize(None)
    facility_launches = facility_launches.sort_values("launch_time_lstd")
    weather = load_best_hourly_weather(rel_path)
    merged = pd.merge_asof(
        facility_launches,
        weather,
        left_on="launch_time_lstd",
        right_on="weather_obs_time_lstd",
        direction="nearest",
        tolerance=pd.Timedelta("2h"),
    )
    merged["weather_matched"] = merged["weather_obs_time_lstd"].notna()
    merged["weather_time_diff_minutes"] = (
        (merged["launch_time_lstd"] - merged["weather_obs_time_lstd"]).dt.total_seconds().abs() / 60
    )
    weather_coverage_rows.append(
        {
            "facility_group": facility,
            "launches": len(facility_launches),
            "matched_launches": int(merged["weather_matched"].sum()),
            "match_rate": merged["weather_matched"].mean(),
            "median_abs_diff_minutes": merged["weather_time_diff_minutes"].median(),
            "weather_start_lstd": weather["weather_obs_time_lstd"].min(),
            "weather_end_lstd": weather["weather_obs_time_lstd"].max(),
        }
    )
    weather_merges.append(merged)

us_launch_weather = pd.concat(weather_merges, ignore_index=True).sort_values("launch_time_utc").reset_index(drop=True)
us_launch_weather["launch_outcome_group"] = us_launch_weather["Launch Status"].where(
    us_launch_weather["Launch Status"] == "Success", "Not Success"
)
us_launch_weather["launch_success_binary"] = (us_launch_weather["Launch Status"] == "Success").astype(int)
us_launch_weather["launch_failure_binary"] = (us_launch_weather["Launch Status"] != "Success").astype(int)
us_launch_weather["is_partial_failure"] = (us_launch_weather["Launch Status"] == "Partial Failure").astype(int)
us_launch_weather["is_failure"] = (us_launch_weather["Launch Status"] == "Failure").astype(int)
us_launch_weather["is_prelaunch_failure"] = (us_launch_weather["Launch Status"] == "Prelaunch Failure").astype(int)
us_launch_weather["rocket_payload_leo"] = pd.to_numeric(us_launch_weather["Rocket Payload to LEO"], errors="coerce")
us_launch_weather["rocket_price_musd"] = pd.to_numeric(us_launch_weather["Rocket Price"], errors="coerce")
us_launch_weather["rocket_price_adjusted_musd"] = pd.to_numeric(us_launch_weather["Rocket Price CPI Adjusted"], errors="coerce")
us_launch_weather["usd_per_kg_leo_adjusted"] = pd.to_numeric(us_launch_weather["USD/kg to LEO CPI Adjusted"], errors="coerce")
top_orgs = us_launch_weather["Rocket Organisation"].value_counts().head(8).index
us_launch_weather["rocket_org_grouped"] = us_launch_weather["Rocket Organisation"].where(
    us_launch_weather["Rocket Organisation"].isin(top_orgs), "Other"
)
us_launch_weather["payload_bin"] = pd.cut(us_launch_weather["rocket_payload_leo"], bins=[0, 500, 2000, 10000, 50000, 500000], include_lowest=True)
season_map = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring", 6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
us_launch_weather["launch_season"] = us_launch_weather["launch_month"].map(season_map)
us_launch_weather["launch_hour_local"] = us_launch_weather["launch_time_lstd"].dt.hour
us_launch_weather["launch_hour_bin"] = pd.cut(us_launch_weather["launch_hour_local"], bins=[-1, 5, 11, 17, 23], labels=["Overnight", "Morning", "Afternoon", "Evening"])
us_launch_weather["era_group"] = pd.cut(
    us_launch_weather["launch_year"],
    bins=[1950, 1979, 1999, 2025],
    labels=["Early space age (1951-1979)", "Transition era (1980-1999)", "Modern era (2000-2024)"],
    include_lowest=True,
)
us_launch_weather["precip_positive_flag"] = us_launch_weather["HourlyPrecipitation"].fillna(0).gt(0)
us_launch_weather["weather_type_reported_flag"] = us_launch_weather["HourlyPresentWeatherType"].fillna("").astype(str).str.len().gt(0)
us_launch_weather["high_wind_flag"] = us_launch_weather["HourlyWindSpeed"].ge(15)
us_launch_weather["low_visibility_flag"] = us_launch_weather["HourlyVisibility"].le(5)
us_launch_weather["high_humidity_flag"] = us_launch_weather["HourlyRelativeHumidity"].ge(80)
us_launch_weather["dewpoint_depression"] = us_launch_weather["HourlyDryBulbTemperature"] - us_launch_weather["HourlyDewPointTemperature"]
us_launch_weather["has_any_precip_signal"] = (
    us_launch_weather[["precip_positive_flag", "present_weather_rain_flag"]]
    .fillna(False).astype(bool).any(axis=1)
)
us_launch_weather["high_wind_and_low_visibility_flag"] = us_launch_weather["high_wind_flag"].fillna(False).astype(bool) & us_launch_weather["low_visibility_flag"].fillna(False).astype(bool)
us_launch_weather["high_wind_and_high_humidity_flag"] = us_launch_weather["high_wind_flag"].fillna(False).astype(bool) & us_launch_weather["high_humidity_flag"].fillna(False).astype(bool)
us_launch_weather["rain_and_low_visibility_flag"] = us_launch_weather["has_any_precip_signal"].fillna(False).astype(bool) & us_launch_weather["low_visibility_flag"].fillna(False).astype(bool)
us_launch_weather["wind_x_visibility"] = us_launch_weather["HourlyWindSpeed"] * us_launch_weather["HourlyVisibility"]
us_launch_weather["wind_x_humidity"] = us_launch_weather["HourlyWindSpeed"] * us_launch_weather["HourlyRelativeHumidity"]
us_launch_weather["weather_match_quality_bin"] = pd.cut(us_launch_weather["weather_time_diff_minutes"], bins=[0, 15, 30, 60, 120], labels=["0-15 min", "15-30 min", "30-60 min", "60-120 min"], include_lowest=True)
us_launch_weather["multi_payload_flag"] = us_launch_weather["payload_count"].fillna(0).gt(1)
us_launch_weather["rocket_price_adjusted_missing_flag"] = us_launch_weather["rocket_price_adjusted_musd"].isna()
us_launch_weather["usd_per_kg_leo_adjusted_missing_flag"] = us_launch_weather["usd_per_kg_leo_adjusted"].isna()
us_launch_weather["config_fairing_diameter_missing_flag"] = us_launch_weather["config_fairing_diameter"].isna()
us_launch_weather["config_fairing_height_missing_flag"] = us_launch_weather["config_fairing_height"].isna()
us_launch_weather["HourlyVisibility_missing_flag"] = us_launch_weather["HourlyVisibility"].isna()
us_launch_weather["HourlyAltimeterSetting_missing_flag"] = us_launch_weather["HourlyAltimeterSetting"].isna()
us_launch_weather["HourlyWetBulbTemperature_missing_flag"] = us_launch_weather["HourlyWetBulbTemperature"].isna()

for group_col, prefix in [("rocket_family", "family"), ("Rocket Organisation", "org"), ("Rocket Name", "config"), ("facility_group", "site")]:
    us_launch_weather = add_prelaunch_history(us_launch_weather, group_col, prefix)

us_launch_weather["family_launches_so_far"] = us_launch_weather["family_prior_launches"]
us_launch_weather["site_launches_so_far"] = us_launch_weather["site_prior_launches"]
us_launch_weather["config_launches_so_far"] = us_launch_weather["config_prior_launches"]
us_launch_weather["org_launches_so_far"] = us_launch_weather["org_prior_launches"]

for raw_col, new_col in [
    ("HourlyWindSpeed", "site_wind_speed_z"),
    ("HourlyVisibility", "site_visibility_z"),
    ("HourlyRelativeHumidity", "site_humidity_z"),
    ("dewpoint_depression", "site_dewpoint_depression_z"),
    ("HourlyAltimeterSetting", "site_altimeter_setting_z"),
]:
    site_mean = us_launch_weather.groupby("facility_group")[raw_col].transform("mean")
    site_std = us_launch_weather.groupby("facility_group")[raw_col].transform("std")
    us_launch_weather[new_col] = (us_launch_weather[raw_col] - site_mean) / site_std.replace(0, np.nan)

weather_merge_coverage = pd.DataFrame(weather_coverage_rows).sort_values("matched_launches", ascending=False).reset_index(drop=True)
feature_engineering_summary = pd.DataFrame(
    {
        "feature_family": [
            "prelaunch historical rates", "launch-history counts", "cadence features",
            "site-relative weather z-scores", "weather interactions", "missingness indicators",
        ],
        "examples": [
            "family_success_rate_prelaunch, org_success_rate_prelaunch, config_success_rate_prelaunch",
            "family_launches_so_far, site_launches_so_far, config_launches_so_far",
            "days_since_previous_launch_site, days_since_previous_launch_family",
            "site_wind_speed_z, site_visibility_z, site_humidity_z",
            "high_wind_and_low_visibility_flag, high_wind_and_high_humidity_flag, rain_and_low_visibility_flag",
            "rocket_price_adjusted_missing_flag, HourlyVisibility_missing_flag, HourlyAltimeterSetting_missing_flag",
        ],
    }
)
feature_engineering_summary


,feature_family,examples
0,prelaunch historical rates,"family_success_rate_prelaunch, org_success_rate_prelaunch, config_success_rate_prelaunch"
1,launch-history counts,"family_launches_so_far, site_launches_so_far, config_launches_so_far"
2,cadence features,"days_since_previous_launch_site, days_since_previous_launch_family"
3,site-relative weather z-scores,"site_wind_speed_z, site_visibility_z, site_humidity_z"
4,weather interactions,"high_wind_and_low_visibility_flag, high_wind_and_high_humidity_flag, rain_and_low_visibility_flag"
5,missingness indicators,"rocket_price_adjusted_missing_flag, HourlyVisibility_missing_flag, HourlyAltimeterSetting_missing_flag"


## 5. Prepared Output Audit



Before exporting anything, this section checks how much weather information is actually available after the merge. That audit matters because some hourly weather fields are much more complete than others, and the modeling notebook needs to know which features are reliable enough to keep.



### What This Audit Cell Does

This cell looks only at launches that successfully matched to weather data and calculates how often each weather feature is populated.

Why this matters:
- A feature can be conceptually useful but still be a poor modeling choice if it is missing for most launches.
- The audit distinguishes between fields that are widely available across the matched sample and fields that may need to be dropped, imputed, or treated carefully.
- This is particularly important for weather text-derived variables and some specialized precipitation fields, which are often less complete than core hourly measurements like temperature or wind speed.


In [6]:
matched_launch_weather = us_launch_weather.loc[us_launch_weather["weather_matched"]].copy()
weather_feature_availability = pd.DataFrame(
    {
        "feature": WEATHER_NUMERIC_COLUMNS + WEATHER_TEXT_COLUMNS + [
            "present_weather_rain_flag", "present_weather_fog_flag",
            "present_weather_thunder_flag", "cloud_cover_broken_or_overcast_flag", "short_duration_precip_max",
        ],
        "non_null_share": [
            matched_launch_weather[col].notna().mean() if col in matched_launch_weather.columns else 0
            for col in WEATHER_NUMERIC_COLUMNS + WEATHER_TEXT_COLUMNS + [
                "present_weather_rain_flag", "present_weather_fog_flag",
                "present_weather_thunder_flag", "cloud_cover_broken_or_overcast_flag", "short_duration_precip_max",
            ]
        ],
    }
).sort_values("non_null_share", ascending=False)
weather_feature_availability


,feature,non_null_share
17,cloud_cover_broken_or_overcast_flag,1.000000
16,present_weather_thunder_flag,1.000000
15,present_weather_fog_flag,1.000000
14,present_weather_rain_flag,1.000000
9,HourlyWindSpeed,0.983688
1,HourlyDryBulbTemperature,0.981560
11,HourlyWindDirection,0.902128
3,HourlyRelativeHumidity,0.901418
2,HourlyDewPointTemperature,0.901418
7,HourlySeaLevelPressure,0.797872


## 6. Write Prepared Outputs



The final section writes both the main prepared dataset and a few diagnostic summaries. Those exports make it easier to reuse the cleaned data in later notebooks without repeating the full preparation pipeline each time.



### What This Export Cell Does

This cell writes the cleaned outputs to `data/derived/`.

What gets exported:
- A site summary describing launch volume, date range, and success rate by normalized facility.
- Merge diagnostics showing how well each facility?s launches matched to weather observations.
- Overall and site-level weather feature availability tables.
- The full launch-level dataset with weather and engineered features, `us_launch_weather_merged.csv`.

Why this matters:
- Exporting these artifacts turns the notebook from a one-off exploration into a reusable preparation step for the rest of the project.
- The diagnostic CSVs make the weather merge transparent, so teammates can inspect coverage and feature completeness without rerunning the notebook or reading all of the preparation code.


In [7]:
us_site_summary_export = (
    us_launches.groupby("facility_group")
    .agg(
        launches=("Launch Id", "count"),
        raw_location_strings=("Location", "nunique"),
        first_launch=("launch_date", "min"),
        last_launch=("launch_date", "max"),
        success_rate=("Launch Status", lambda s: (s == "Success").mean()),
    )
    .reset_index()
    .sort_values("launches", ascending=False)
)

facility_feature_availability_export = (
    matched_launch_weather.groupby("facility_group")[
        [c for c in WEATHER_NUMERIC_COLUMNS if c in matched_launch_weather.columns]
    ]
    .agg(lambda s: s.notna().mean())
    .T
    .reset_index()
    .rename(columns={"index": "feature"})
)

us_site_summary_export.to_csv(OUTPUT_DIR / "us_launch_site_summary.csv", index=False)
weather_merge_coverage.to_csv(OUTPUT_DIR / "weather_merge_coverage.csv", index=False)
weather_feature_availability.to_csv(OUTPUT_DIR / "weather_feature_availability.csv", index=False)
facility_feature_availability_export.to_csv(OUTPUT_DIR / "facility_weather_feature_availability.csv", index=False)
us_launch_weather.to_csv(OUTPUT_DIR / "us_launch_weather_merged.csv", index=False)

pd.DataFrame(
    {
        "file": [
            str(OUTPUT_DIR / "us_launch_site_summary.csv"),
            str(OUTPUT_DIR / "weather_merge_coverage.csv"),
            str(OUTPUT_DIR / "weather_feature_availability.csv"),
            str(OUTPUT_DIR / "facility_weather_feature_availability.csv"),
            str(OUTPUT_DIR / "us_launch_weather_merged.csv"),
        ]
    }
)


,file
0,data\derived\us_launch_site_summary.csv
1,data\derived\weather_merge_coverage.csv
2,data\derived\weather_feature_availability.csv
3,data\derived\facility_weather_feature_availability.csv
4,data\derived\us_launch_weather_merged.csv
